# 1) IMPORTAR LIBRERIAS

In [27]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 1.2) DISCLAIMER PARAGUAY Y URUGUAY

In [28]:
# ===================================================================
# DISCLAIMER ESPECIAL: PARAGUAY Y URUGUAY → OTHER COUNTRIES
# ===================================================================

def aplicar_disclaimer_pais(df, columna_pais):
    """
    DISCLAIMER ESPECIAL PARA RECONOCIMIENTO DE INGRESOS:
    Paraguay y Uruguay utilizarán factores de 'Other Countries'
    
    Args:
        df: DataFrame a procesar
        columna_pais: Nombre de la columna que contiene los países
    
    Returns:
        DataFrame con disclaimer aplicado
    """
    df_copy = df.copy()
    
    # Mapping de países con disclaimer
    disclaimer_mapping = {
        'Paraguay': 'Other Countries',
        'Uruguay': 'Other Countries'
    }
    
    # Aplicar mapping
    df_copy[columna_pais] = df_copy[columna_pais].replace(disclaimer_mapping)
    
    return df_copy


# 2) IMPORTAR PROYECCIONES Y FACTOR DE DISTRIBUCIÓN DIARIA

In [29]:
# CONFIGURAR RUTAS DE ARCHIVOS
archivo_factores_gd_b2b2c = r"C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Estacionalidad Diaria/Factor diario GD - WLs.xlsx"
#archivo_factores_ri_b2b = r"C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Estacionalidad Diaria/Factor diario RI - B2B.xlsx"
archivo_factores_gd_b2b = r"C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Estacionalidad Diaria/Factor diario GD - B2B.xlsx"
archivo_wl = "C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Run Rate/2026.09.14 - W37/Inputs Python/WLs - Modelo Run Rate1.xlsx"
archivo_api = "C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Run Rate/2026.09.14 - W37/Inputs Python/API - Modelo Run Rate1.xlsx"
archivo_html = "C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Run Rate/2026.09.14 - W37/Inputs Python/HTML - Modelo Run Rate1.xlsx"
print("🚀 DISTRIBUCIÓN DIARIA DE P&L - MULTI NEGOCIO")
print("=" * 70)
print(f"📂 Archivo WL/B2B2C: {archivo_wl}")
print(f"📂 Archivo API/B2B-MAY: {archivo_api}")
print(f"📂 Archivo HTML/B2B-MIN: {archivo_html}")
print(f"📂 Archivo Factores GD-B2B2C: {archivo_factores_gd_b2b2c}")
#print(f"📂 Archivo Factores RI-B2B: {archivo_factores_ri_b2b}")
print(f"📂 Archivo Factores GD-B2B: {archivo_factores_gd_b2b}")

🚀 DISTRIBUCIÓN DIARIA DE P&L - MULTI NEGOCIO
📂 Archivo WL/B2B2C: C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Run Rate/2026.09.14 - W37/Inputs Python/WLs - Modelo Run Rate1.xlsx
📂 Archivo API/B2B-MAY: C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Run Rate/2026.09.14 - W37/Inputs Python/API - Modelo Run Rate1.xlsx
📂 Archivo HTML/B2B-MIN: C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Run Rate/2026.09.14 - W37/Inputs Python/HTML - Modelo Run Rate1.xlsx
📂 Archivo Factores GD-B2B2C: C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Estacionalidad Diaria/Factor diario GD - WLs.xlsx
📂 Archivo Factores GD-B2B: C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Estacionalidad Diaria/Factor diario GD - B2B.xlsx


# 3) CONFIGURAR MAPEOS

In [30]:
# 3) CONFIGURAR MAPEOS
#CONFIGURACIÓN DE MAPEO: Negocio -> Archivo -> Factor
configuracion_negocios = {
    'B2B2C': {
        'archivo': archivo_wl,
        'hoja_proyecciones': 'P&L',
        'hoja_factor': 'Factores B2B2C',
        'archivo_factor': archivo_factores_gd_b2b2c,  # ✅ GD para B2B2C
        'descripcion': 'White Labels / B2B2C',
        'nombre_corto': 'WL_B2B2C'
    },
    'B2B_MAY': {
        'archivo': archivo_api,
        'hoja_proyecciones': 'P&L Emision',
        'hoja_factor': 'Factores B2B-MAY',
        'archivo_factor': archivo_factores_gd_b2b,  # ✅ GD para B2B-MAY
        'descripcion': 'API / B2B-MAY',
        'nombre_corto': 'API'
    },
    'B2B_MIN': {
        'archivo': archivo_html,
        'hoja_proyecciones': 'P&L Emision',
        'hoja_factor': 'Factores B2B-MIN',
        'archivo_factor': archivo_factores_gd_b2b,  # ✅ GD para B2B-MIN
        'descripcion': 'HTML / B2B-MIN',
        'nombre_corto': 'HTML'
    }
}
print("\n📋 CONFIGURACIÓN DE NEGOCIOS:")
print("-" * 50)
for negocio, config in configuracion_negocios.items():
    factor_tipo = "GD" if "gd" in config['archivo_factor'] else "ESTÁNDAR"
    print(f"  {negocio}: {config['descripcion']}")
    print(f"    📊 Proyecciones: {config['hoja_proyecciones']}")
    print(f"    🔗 Factor: {config['hoja_factor']}")
    print(f"    🎯 Archivo Factor: {factor_tipo}")



📋 CONFIGURACIÓN DE NEGOCIOS:
--------------------------------------------------
  B2B2C: White Labels / B2B2C
    📊 Proyecciones: P&L
    🔗 Factor: Factores B2B2C
    🎯 Archivo Factor: ESTÁNDAR
  B2B_MAY: API / B2B-MAY
    📊 Proyecciones: P&L Emision
    🔗 Factor: Factores B2B-MAY
    🎯 Archivo Factor: ESTÁNDAR
  B2B_MIN: HTML / B2B-MIN
    📊 Proyecciones: P&L Emision
    🔗 Factor: Factores B2B-MIN
    🎯 Archivo Factor: ESTÁNDAR


# 4) CARGAR Y PREPARAR FACTORES DIARIOS

In [31]:
print("\n📊 CARGANDO FACTORES DIARIOS...")
print("-" * 40)

factores_por_negocio = {}

for negocio, config in configuracion_negocios.items():
    try:
        print(f"  🔄 Cargando {config['hoja_factor']}...")
        print(f"    📁 Archivo: {config['archivo_factor']}")
        df_factor = pd.read_excel(config['archivo_factor'], sheet_name=config['hoja_factor'])
        print(f"    📊 Dimensiones originales: {df_factor.shape}")
        
        # 🔍 DIAGNÓSTICO INICIAL
        print(f"    📋 Columnas en archivo: {list(df_factor.columns)}")

        # 🔍 VERIFICAR COLUMNAS ANTES DE LIMPIAR
        columnas_requeridas = ['pais', 'viaje', 'anio', 'Factor combinado']
        columnas_faltantes = [col for col in columnas_requeridas if col not in df_factor.columns]
        
        if columnas_faltantes:
            print(f"    ❌ Columnas faltantes para limpieza: {columnas_faltantes}")
            factores_por_negocio[negocio] = None
            continue

        # Limpiar y preparar factores
        print(f"    🧹 Limpiando registros con valores nulos...")
        print(f"       Antes: {df_factor.shape[0]:,} registros")
        df_factor_clean = df_factor.dropna(subset=['pais', 'viaje', 'Factor combinado'])
        print(f"       Después: {df_factor_clean.shape[0]:,} registros")
        
        # 🔍 VERIFICAR SI QUEDÓ VACÍO
        if len(df_factor_clean) == 0:
            print(f"    ❌ No quedan registros después de limpiar")
            factores_por_negocio[negocio] = None
            continue
        
        # 🔍 DIAGNÓSTICO DETALLADO DE COLUMNAS
        print(f"    🔍 Diagnóstico de columnas para merge:")
        columnas_para_merge = ['fecha', 'mes', 'anio', 'pais', 'viaje', 'Factor combinado']
        columnas_disponibles_merge = []
        
        for col in columnas_para_merge:
            if col in df_factor_clean.columns:
                print(f"       ✅ '{col}': EXISTE")
                columnas_disponibles_merge.append(col)
            else:
                print(f"       ❌ '{col}': NO EXISTE")
                # Buscar columnas similares
                similares = [c for c in df_factor_clean.columns if col.lower() in c.lower() or c.lower() in col.lower()]
                if similares:
                    print(f"          🔍 Similares: {similares}")
        
        # ✅ LÓGICA ESPECIAL PARA B2B2C CON PARTNER
        if negocio == 'B2B2C' and 'partner' in df_factor_clean.columns:
            print(f"    🎯 B2B2C: Incluyendo campo Partner")
            columnas_para_merge_final = ['fecha', 'mes', 'anio', 'pais', 'viaje', 'partner', 'Factor combinado']
            if all(col in df_factor_clean.columns for col in columnas_para_merge_final):
                df_factor_merge = df_factor_clean[columnas_para_merge_final].copy()
                print(f"    ✅ Merge B2B2C con Partner configurado")
            else:
                faltantes = [col for col in columnas_para_merge_final if col not in df_factor_clean.columns]
                print(f"    ❌ Columnas faltantes para B2B2C: {faltantes}")
                factores_por_negocio[negocio] = None
                continue
        else:
            # Para otros negocios, lógica original
            print(f"    📊 {negocio}: Sin campo Partner")
            if all(col in df_factor_clean.columns for col in columnas_para_merge):
                df_factor_merge = df_factor_clean[columnas_para_merge].copy()
                print(f"    ✅ Merge estándar configurado")
            else:
                faltantes = [col for col in columnas_para_merge if col not in df_factor_clean.columns]
                print(f"    ❌ Columnas faltantes: {faltantes}")
                factores_por_negocio[negocio] = None
                continue
        
        # Agregar columna de día
        df_factor_merge['Dia'] = df_factor_merge['fecha'].dt.day
        
        factores_por_negocio[negocio] = df_factor_merge
        
        print(f"    ✅ {df_factor_merge.shape[0]:,} registros finales")
        print(f"    📈 Factor rango: {df_factor_clean['Factor combinado'].min():.4f} - {df_factor_clean['Factor combinado'].max():.4f}")
        print(f"    📋 Columnas finales: {list(df_factor_merge.columns)}")
        
    except Exception as e:
        print(f"    ❌ Error cargando {config['hoja_factor']}: {e}")
        print(f"       Archivo: {config['archivo_factor']}")
        import traceback
        print(f"       Detalle: {traceback.format_exc()}")
        factores_por_negocio[negocio] = None

print(f"\n✅ Factores cargados para {sum(1 for f in factores_por_negocio.values() if f is not None)}/{len(configuracion_negocios)} negocios")


📊 CARGANDO FACTORES DIARIOS...
----------------------------------------
  🔄 Cargando Factores B2B2C...
    📁 Archivo: C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Estacionalidad Diaria/Factor diario GD - WLs.xlsx
    📊 Dimensiones originales: (25785, 27)
    📋 Columnas en archivo: ['fecha', 'mes_proyectado', 'mes', 'anio', 'Cantidad dias mes', 'Dia semana', 'Nombre', 'Cant. Dia en la Sem', 'Cantidad dias semana', 'Tipo dia', 'Tipo dia N', 'Cantidad tipo dia en el mes', 'lob', 'pais', 'partner', 'viaje', 'GB', 'Factor dia semana', 'Factor dia semana x Q dias semana', 'Factor 1', 'Factor tipo dia', 'Factor tipo dia x Q dias', 'Factor 2', 'Factor combinado', 'Rdo Factor 1', 'Rdo Facto 2', 'Rdo Combinado']
    🧹 Limpiando registros con valores nulos...
       Antes: 25,785 registros
       Después: 25,750 registros
    🔍 Diagnóstico de columnas para merge:
       ✅ 'fecha': EXISTE
       ✅ 'mes': EXISTE
       ✅ 'anio': EXISTE
    

# 5) PROCESAR NEGOCIOS

In [32]:
# ===================================================================
# FUNCIÓN PARA PROCESAR CADA NEGOCIO
# ===================================================================

def procesar_negocio(negocio, config):
    """
    Procesa un archivo de proyecciones con sus factores correspondientes
    """
    print(f"\n🔄 PROCESANDO: {config['descripcion']}")
    print("-" * 50)
    
    if factores_por_negocio[negocio] is None:
        print(f"  ❌ No hay factores disponibles para {negocio}")
        return None
    
    try:
        # CARGAR PROYECCIONES
        print(f"  📈 Cargando proyecciones desde '{config['hoja_proyecciones']}'...")
        df_proyecciones = pd.read_excel(config['archivo'], sheet_name=config['hoja_proyecciones'])
        print(f"    ✅ Dimensiones: {df_proyecciones.shape}")
        

        # MAPEO DE LOB_CANAL: API → B2B-MAY
        if 'lob_canal' in df_proyecciones.columns:
            mapeo_lob_canal = {'API': 'B2B-MAY', 'HTML': 'B2B-MIN'}
            df_proyecciones['lob_canal'] = df_proyecciones['lob_canal'].replace(mapeo_lob_canal)
            print(f"    🔄 Mapeo lob_canal aplicado: API → B2B-MAY")

        # 🔍 DIAGNÓSTICO DE COLUMNAS CRÍTICAS
        print(f"\n  🔍 DIAGNÓSTICO DE COLUMNAS CRÍTICAS:")
        print(f"    📊 Total registros: {len(df_proyecciones):,}")
        print(f"    📋 Columnas disponibles: {list(df_proyecciones.columns)}")
        if 'pais' in df_proyecciones.columns:
            print(f"    🌍 Valores únicos en 'pais': {df_proyecciones['pais'].unique()[:10]}")
            print(f"    ❌ Registros con país nulo: {df_proyecciones['pais'].isna().sum():,}")
            print(f"    📝 Tipo de dato pais: {df_proyecciones['pais'].dtype}")
            print(f"    🔍 Muestra valores pais: {df_proyecciones['pais'].head(10).tolist()}")
        else:
            print(f"    ❌ Columna 'pais' NO EXISTE")
            
        if 'viaje' in df_proyecciones.columns:
            print(f"    🎯 Valores únicos en 'viaje': {df_proyecciones['viaje'].unique()[:10]}")
            print(f"    ❌ Registros con viaje nulo: {df_proyecciones['viaje'].isna().sum():,}")
            print(f"    📝 Tipo de dato viaje: {df_proyecciones['viaje'].dtype}")
            print(f"    🔍 Muestra valores viaje: {df_proyecciones['viaje'].head(10).tolist()}")
        else:
            print(f"    ❌ Columna 'viaje' NO EXISTE")

        # CREAR MAPEO DE MESES COMPLETO
        mapeo_meses = {
            'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 'Mayo': 5, 'Junio': 6,
            'Julio': 7, 'Agosto': 8, 'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12,
            1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12
        }
        
        # LIMPIAR REGISTROS CON PAÍS O VIAJE NULOS
        print(f"\n  🧹 LIMPIANDO REGISTROS CON PAÍS O VIAJE NULOS...")
        print(f"    📊 Antes de limpiar: {len(df_proyecciones):,} registros")
        df_pl_clean = df_proyecciones.dropna(subset=['pais', 'viaje']).copy()
        print(f"    📊 Después de limpiar: {len(df_pl_clean):,} registros")
        print(f"    🗑️  Registros eliminados: {len(df_proyecciones) - len(df_pl_clean):,}")

        # ⬇️ NUEVO: aplicar disclaimer ANTES del merge con factores
        # Paraguay y Uruguay pasan a ser 'Other Countries' para que el merge los encuentre
        df_pl_clean = aplicar_disclaimer_pais(df_pl_clean, 'pais')
        print(f"    ✅ Disclaimer aplicado: Paraguay/Uruguay → 'Other Countries'")
        
        # VERIFICACIÓN DE SEGURIDAD
        if len(df_pl_clean) == 0:
            print(f"    ❌ SIN REGISTROS DESPUÉS DE LIMPIAR")
            return None
        
        # Detectar columna de mes (mes_ri primero)
        columna_mes = None
        for col in ['mes_venta','mes_ri', 'mes', 'mes_proyectado', 'Fecha Mes']:
            if col in df_pl_clean.columns:
                columna_mes = col
                break
        
        if columna_mes is None:
            print(f"    ❌ No se encuentra columna de mes")
            return None
        
        # APLICAR MAPEO DE MESES
        df_pl_clean['Mes_Numerico'] = df_pl_clean[columna_mes].map(mapeo_meses)
        df_pl_clean = df_pl_clean.dropna(subset=['Mes_Numerico'])
        
        # VERIFICAR COLUMNA AÑO — crear desde anio_ri
        if 'anio_venta' in df_pl_clean.columns:
            df_pl_clean['anio'] = df_pl_clean['anio_venta']    # API y HTML
        elif 'anio_ri' in df_pl_clean.columns:
            df_pl_clean['anio'] = df_pl_clean['anio_ri']       # WLs (B2B2C)
        elif 'anio' not in df_pl_clean.columns:
            print(f"    ❌ No se encuentra columna de año (ni 'anio_venta' ni 'anio_ri')")
            return None
      
        print(f"    ✅ Columna 'anio' configurada correctamente")
        print(f"    📅 Años únicos en proyecciones: {sorted(df_pl_clean['anio'].unique())}")
        print(f"    ✅ Después de limpiar: {df_pl_clean.shape}")
        print(f"    🔗 Usando columna de mes: '{columna_mes}'")
        
        # DEFINIR MÉTRICAS A DISTRIBUIR
        metricas_base = [
            'orders',
            'gross_bookings',
            'up_front_incentives',
            'fees',
            'commercial_discounts',
            'cancellations',
            'cost_of_installments',
            'credit_card_processing',
            'white_labels_api',
            'affiliates',
            'income_from_outsourced_services',
            'back_end_incentives',
            'other_incentives',
            'breakage_revenue',
            'media_revenue',
            'revenue_tax',
            'customer_service',
            'errors',
            'frauds',
            'customer_claims',
            'intercompany_usd',
            'operations',
            'other_transactional_taxes',
            'vendor_commissions',
            'dif_fx',
            'currency_hedge',
            'financial_results'
            # net_revenue y fvm se calculan al final, no se distribuyen
        ]
        
        metricas_validas = [col for col in metricas_base if col in df_pl_clean.columns]
        print(f"    📊 Métricas encontradas: {len(metricas_validas)}/{len(metricas_base)}")
        if len(metricas_validas) <= 10:
            print(f"        {metricas_validas}")
        else:
            print(f"        Primeras 10: {metricas_validas[:10]}")
        
        # OBTENER FACTORES PARA ESTE NEGOCIO
        df_factor_merge = factores_por_negocio[negocio]
        
        # VERIFICAR AÑOS EN FACTORES
        print(f"  📅 DIAGNÓSTICO DE AÑOS EN FACTORES:")
        if 'anio' in df_factor_merge.columns:
            print(f"    Factores - años únicos: {sorted(df_factor_merge['anio'].unique())}")
        else:
            print(f"    ❌ Factores NO tienen columna 'anio'")
            return None
        
        # DIAGNÓSTICO CRÍTICO DE MESES
        print(f"\n  🗓️  DIAGNÓSTICO CRÍTICO DE MESES:")
        print(f"    📅 PROYECCIONES:")
        print(f"       Meses únicos en '{columna_mes}': {sorted(df_pl_clean[columna_mes].unique())}")
        print(f"       Meses únicos en 'Mes_Numerico': {sorted(df_pl_clean['Mes_Numerico'].unique())}")
        print(f"       Años únicos: {sorted(df_pl_clean['anio'].unique())}")
        print(f"    📅 FACTORES:")
        print(f"       Meses únicos en factores: {sorted(df_factor_merge['mes'].unique())}")
        print(f"       Años únicos en factores: {sorted(df_factor_merge['anio'].unique())}")
        
        combo_proyecciones = set(zip(df_pl_clean['Mes_Numerico'], df_pl_clean['anio']))
        combo_factores = set(zip(df_factor_merge['mes'], df_factor_merge['anio']))
        print(f"    🔗 COMBINACIONES MES-AÑO:")
        print(f"       En proyecciones: {sorted(combo_proyecciones)}")
        print(f"       En factores: {sorted(combo_factores)}")
        print(f"       Intersección: {sorted(combo_proyecciones.intersection(combo_factores))}")
        print(f"       Sin match proyecciones: {sorted(combo_proyecciones - combo_factores)}")
        print(f"       Sin match factores: {sorted(combo_factores - combo_proyecciones)}")
        
        if negocio == 'B2B2C':
            print(f"    💰 VALORES POR MES EN PROYECCIONES (B2B2C):")
            if 'gross_bookings' in df_pl_clean.columns:
                valores_por_mes = df_pl_clean.groupby(['Mes_Numerico', 'anio'])['gross_bookings'].sum()
                for (mes, anio), valor in valores_por_mes.items():
                    print(f"       Mes {mes} - {anio}: {valor:,.2f}")
        
        # CONVERSIÓN DE TIPOS PARA MERGE CORRECTO
        print(f"\n  🔧 CONVIRTIENDO TIPOS DE DATOS PARA MERGE:")
        df_pl_clean['Mes_Numerico'] = df_pl_clean['Mes_Numerico'].astype(int)
        df_pl_clean['anio'] = df_pl_clean['anio'].astype(int)
        df_factor_merge['mes'] = df_factor_merge['mes'].astype(int)
        df_factor_merge['anio'] = df_factor_merge['anio'].astype(int)
        print(f"    ✅ Tipos convertidos a int para merge exacto")
        print(f"    📊 Proyecciones - Mes_Numerico: {df_pl_clean['Mes_Numerico'].dtype}, anio: {df_pl_clean['anio'].dtype}")
        print(f"    📊 Factores - mes: {df_factor_merge['mes'].dtype}, anio: {df_factor_merge['anio'].dtype}")
        
        # VERIFICACIÓN PRE-MERGE
        print(f"\n  🔍 VERIFICACIÓN PRE-MERGE:")
        print(f"    📊 Registros únicos en proyecciones: {len(df_pl_clean):,}")
        print(f"    📊 Registros únicos en factores: {len(df_factor_merge):,}")
        if negocio == 'B2B2C':
            combo_proj_unicas = df_pl_clean[['Mes_Numerico', 'anio', 'pais', 'partner', 'viaje']].drop_duplicates()
            combo_fact_unicas = df_factor_merge[['mes', 'anio', 'pais', 'partner', 'viaje']].drop_duplicates()
            print(f"    🔑 Combinaciones únicas proyecciones: {len(combo_proj_unicas):,}")
            print(f"    🔑 Combinaciones únicas factores: {len(combo_fact_unicas):,}")
        
        # ===================================================================
        # LÓGICA DE MERGE
        # ===================================================================
        if negocio == 'B2B2C' and 'partner' in df_factor_merge.columns and 'partner' in df_pl_clean.columns:
            print(f"  🎯 Realizando merge especial B2B2C con Partner...")
            
            # PASO 1: Merge con partner específico
            df_merged_paso1 = df_pl_clean.merge(
                df_factor_merge,
                left_on=['Mes_Numerico', 'anio', 'pais', 'partner', 'viaje'],
                right_on=['mes', 'anio', 'pais', 'partner', 'viaje'],
                how='left',
                suffixes=('', '_factor')
            )
            
            sin_factor_paso1 = df_merged_paso1['Factor combinado'].isna()
            registros_sin_factor = sin_factor_paso1.sum()
            print(f"    ✅ Paso 1 - Match específico por Partner: {len(df_merged_paso1) - registros_sin_factor:,}/{len(df_merged_paso1):,}")
            
            if registros_sin_factor > 0:
                print(f"    🔄 Paso 2 - Fallback con Partner='Todos' para {registros_sin_factor:,} registros...")
                
                df_factor_todos = df_factor_merge[df_factor_merge['partner'] == 'Todos'].copy()
                
                if not df_factor_todos.empty:
                    registros_sin_factor_df = df_merged_paso1[sin_factor_paso1].copy()
                    
                    # Preservar partner original
                    registros_sin_factor_df['partner_original'] = registros_sin_factor_df['partner']
                    
                    # Limpiar columnas del merge fallido
                    cols_a_limpiar = [col for col in ['fecha_factor', 'mes_factor',
                                                      'anio_factor', 'partner_factor', 'Factor combinado']
                                      if col in registros_sin_factor_df.columns]
                    if cols_a_limpiar:
                        registros_sin_factor_df = registros_sin_factor_df.drop(columns=cols_a_limpiar)
                        print(f"    🧹 Columnas limpiadas: {cols_a_limpiar}")
                    
                    # Columnas disponibles en factor Todos
                    columnas_base_factores = ['mes', 'anio', 'pais', 'viaje', 'Factor combinado']
                    columnas_factores_disponibles = columnas_base_factores.copy()
                    if 'fecha' in df_factor_todos.columns:
                        columnas_factores_disponibles.append('fecha')
                    print(f"    📋 Columnas disponibles en factores: {columnas_factores_disponibles}")
                    
                    # PASO 2: Merge con partner = Todos
                    registros_con_factor_paso2 = registros_sin_factor_df.merge(
                        df_factor_todos[columnas_factores_disponibles],
                        left_on=['Mes_Numerico', 'anio', 'pais', 'viaje'],
                        right_on=['mes', 'anio', 'pais', 'viaje'],
                        how='left',
                        suffixes=('', '_todos')
                    )
                    
                    # ✅ CORRECCIÓN — recuperar fecha desde factor Todos cuando fecha es nula
                    if 'fecha_todos' in registros_con_factor_paso2.columns:
                        mask_fecha_null = registros_con_factor_paso2['fecha'].isna()
                        registros_con_factor_paso2.loc[mask_fecha_null, 'fecha'] = \
                            registros_con_factor_paso2.loc[mask_fecha_null, 'fecha_todos']
                        print(f"    📅 Fecha recuperada desde factor Todos: {mask_fecha_null.sum():,} registros")
                    
                    # Restaurar partner original
                    if 'partner_original' in registros_con_factor_paso2.columns:
                        registros_con_factor_paso2['partner'] = registros_con_factor_paso2['partner_original']
                        registros_con_factor_paso2 = registros_con_factor_paso2.drop(columns=['partner_original'])
                    
                    # Preservar fecha antes de alineación
                    fecha_paso2_exists = 'fecha' in registros_con_factor_paso2.columns
                    if fecha_paso2_exists:
                        registros_con_factor_paso2['fecha_paso2_temp'] = registros_con_factor_paso2['fecha']
                        print(f"    📅 Preservando fecha del paso 2")
                    
                    registros_con_factor_paso1 = df_merged_paso1[~sin_factor_paso1].copy()
                    
                    # Alinear columnas entre paso1 y paso2
                    cols_paso1 = set(registros_con_factor_paso1.columns)
                    cols_paso2 = set(registros_con_factor_paso2.columns)
                    
                    for col in cols_paso1 - cols_paso2:
                        if col.endswith('_factor') or col.endswith('_todos') or col.endswith('_temp'):
                            continue
                        if col == 'fecha' and fecha_paso2_exists:
                            continue
                        registros_con_factor_paso2[col] = None
                        print(f"    ➕ Agregando columna faltante en paso2: {col}")
                    
                    for col in cols_paso2 - cols_paso1:
                        if col.endswith('_factor') or col.endswith('_todos') or col.endswith('_temp'):
                            continue
                        registros_con_factor_paso1[col] = None
                        print(f"    ➕ Agregando columna faltante en paso1: {col}")
                    
                    # Restaurar fecha preservada
                    if fecha_paso2_exists and 'fecha_paso2_temp' in registros_con_factor_paso2.columns:
                        registros_con_factor_paso2['fecha'] = registros_con_factor_paso2['fecha_paso2_temp']
                        registros_con_factor_paso2 = registros_con_factor_paso2.drop(columns=['fecha_paso2_temp'])
                        print(f"    📅 Fecha restaurada en paso 2")
                    
                    # Limpiar columnas auxiliares
                    for df_temp in [registros_con_factor_paso1, registros_con_factor_paso2]:
                        cols_aux = [col for col in df_temp.columns if col.endswith('_factor') or col.endswith('_todos') or col.endswith('_temp')]
                        if cols_aux:
                            df_temp.drop(columns=cols_aux, inplace=True, errors='ignore')
                            print(f"    🧹 Limpiando columnas auxiliares: {cols_aux}")
                    
                    # Concatenar paso1 y paso2
                    common_cols = sorted(set(registros_con_factor_paso1.columns) & set(registros_con_factor_paso2.columns))
                    print(f"    🔗 Columnas comunes para concat: {len(common_cols)}")
                    
                    if 'fecha' in common_cols:
                        fecha_nulos_paso1 = registros_con_factor_paso1['fecha'].isna().sum()
                        fecha_nulos_paso2 = registros_con_factor_paso2['fecha'].isna().sum()
                        print(f"    📅 Fechas nulas - Paso1: {fecha_nulos_paso1}, Paso2: {fecha_nulos_paso2}")
                    
                    df_merged = pd.concat([
                        registros_con_factor_paso1[common_cols],
                        registros_con_factor_paso2[common_cols]
                    ], ignore_index=True)
                    
                    matches_paso2 = (~registros_con_factor_paso2['Factor combinado'].isna()).sum()
                    print(f"    ✅ Paso 2 - Match con 'Todos': {matches_paso2:,}/{len(registros_con_factor_paso2):,}")
                    print(f"    🔑 Partners originales preservados en paso 2")
                    print(f"    📅 Fechas preservadas en paso 2")
                else:
                    print(f"    ⚠️  No se encontraron factores con Partner='Todos'")
                    df_merged = df_merged_paso1
            else:
                df_merged = df_merged_paso1
                
        else:
            # MERGE ESTÁNDAR PARA API Y HTML
            print(f"  🔗 Realizando merge por Mes + Año + País + Viaje...")
            df_merged = df_pl_clean.merge(
                df_factor_merge,
                left_on=['Mes_Numerico', 'anio', 'pais', 'viaje'],
                right_on=['mes', 'anio', 'pais', 'viaje'],
                how='left'
            )
        
        print(f"    ✅ Registros después del merge: {len(df_merged):,}")
        
        # VERIFICACIÓN POST-MERGE
        print(f"\n  🔍 VERIFICACIÓN POST-MERGE:")
        
        if 'mes' in df_merged.columns:
            nulos_mes = df_merged['mes'].isna().sum()
            print(f"    ✅ Registros con mes válido: {len(df_merged) - nulos_mes:,}/{len(df_merged):,}")
            if nulos_mes > 0:
                print(f"    ⚠️  Registros con mes nulo: {nulos_mes:,}")
            else:
                print(f"    🎉 ÉXITO: Todos los registros tienen mes válido")
        
        if 'partner' in df_merged.columns:
            nulos_partner = df_merged['partner'].isna().sum()
            print(f"    ✅ Registros con partner válido: {len(df_merged) - nulos_partner:,}/{len(df_merged):,}")
            if nulos_partner > 0:
                print(f"    ⚠️  Registros con partner nulo: {nulos_partner:,}")
            else:
                print(f"    🎉 ÉXITO: Todos los registros tienen partner válido")
            partner_counts = df_merged['partner'].value_counts(dropna=False)
            print(f"    📊 Distribución de partners (top 10):")
            for partner, count in partner_counts.head(10).items():
                print(f"       {partner}: {count:,} registros")
            if len(partner_counts) > 10:
                print(f"       ... y {len(partner_counts) - 10} partners más")
        
        if 'fecha' in df_merged.columns:
            nulos_fecha = df_merged['fecha'].isna().sum()
            print(f"    ✅ Registros con fecha válida: {len(df_merged) - nulos_fecha:,}/{len(df_merged):,}")
            if nulos_fecha > 0:
                print(f"    ⚠️  Registros con fecha nula: {nulos_fecha:,}")
            else:
                print(f"    🎉 ÉXITO: Todos los registros tienen fecha válida")
        else:
            print(f"    ❌ PROBLEMA: No existe columna 'fecha' en el resultado")
        
        if negocio == 'B2B2C' and 'gross_bookings' in df_merged.columns and 'mes' in df_merged.columns:
            print(f"    💰 VERIFICACIÓN VALORES POST-MERGE:")
            df_validos = df_merged[df_merged['mes'].notna()]
            if len(df_validos) > 0:
                valores_post_merge = df_validos.groupby(['mes', 'anio'])['gross_bookings'].sum()
                for (mes, anio), valor in valores_post_merge.items():
                    print(f"       Mes {mes} - {anio}: {valor:,.2f}")
        
        if len(df_pl_clean) > 0:
            print(f"    📈 Factor expansión: {len(df_merged) / len(df_pl_clean):.1f}x")
        
        sin_factor = df_merged['Factor combinado'].isna().sum()
        cobertura = ((len(df_merged) - sin_factor) / len(df_merged) * 100) if len(df_merged) > 0 else 0
        print(f"    📋 Cobertura final: {cobertura:.1f}% ({len(df_merged) - sin_factor:,}/{len(df_merged):,})")
        if sin_factor > 0:
            print(f"    ⚠️  {sin_factor:,} registros sin factor")
        
        # MUESTRA DEL MERGE
        print(f"\n  📊 MUESTRA DEL MERGE:")
        cols_merge = ['fecha', columna_mes, 'mes', 'anio', 'pais', 'partner', 'viaje']
        if metricas_validas:
            cols_merge.extend(metricas_validas[:2])
        cols_merge.append('Factor combinado')
        cols_disponibles = [col for col in cols_merge if col in df_merged.columns]
        if cols_disponibles:
            print(df_merged[cols_disponibles].head(3).to_string(index=False))
        
        # APLICAR DISTRIBUCIÓN A TODAS LAS MÉTRICAS
        print(f"\n  💰 APLICANDO DISTRIBUCIÓN DIARIA A MÉTRICAS:")
        print(f"    📊 Métricas a distribuir: {len(metricas_validas)}")
        for metrica in metricas_validas:
            if metrica in df_merged.columns and 'Factor combinado' in df_merged.columns:
                try:
                    df_merged[metrica] = pd.to_numeric(df_merged[metrica], errors='coerce')
                    factor_combinado = pd.to_numeric(df_merged['Factor combinado'], errors='coerce')
                    mask_validos = df_merged[metrica].notna() & factor_combinado.notna()
                    df_merged.loc[mask_validos, metrica] = df_merged.loc[mask_validos, metrica] * factor_combinado[mask_validos]
                except Exception as e:
                    print(f"    ⚠️  Error distribuyendo {metrica}: {str(e)}")
                    continue
        print(f"    ✅ Distribución aplicada a métricas")
        
        # Agregar metadatos
        df_merged['Negocio'] = negocio
        df_merged['Tipo_Negocio'] = config['nombre_corto']
        df_merged['Metricas_Validas'] = len(metricas_validas)
        
        # VERIFICACIÓN FINAL DE INTEGRIDAD
        print(f"\n  🔍 VERIFICACIÓN FINAL DE INTEGRIDAD:")
        print(f"    📊 Registros finales: {len(df_merged):,}")
        print(f"    📋 Columnas finales: {len(df_merged.columns)}")
        campos_criticos = ['mes', 'anio', 'pais', 'viaje', 'partner', 'fecha', 'Factor combinado']
        campos_disponibles = [campo for campo in campos_criticos if campo in df_merged.columns]
        if campos_disponibles:
            registros_completos = df_merged[campos_disponibles].notna().all(axis=1).sum()
            print(f"    ✅ Registros con datos críticos completos: {registros_completos:,}/{len(df_merged):,}")
            if registros_completos < len(df_merged):
                print(f"    ⚠️  Registros con datos críticos faltantes: {len(df_merged) - registros_completos:,}")
        
        return {
            'data': df_merged,
            'metricas': metricas_validas,
            'config': config
        }
        
    except Exception as e:
        print(f"  ❌ Error procesando {negocio}: {e}")
        import traceback
        print(f"     Detalle: {traceback.format_exc()}")
        return None

In [33]:
##df_test = pd.read_excel(archivo_api, sheet_name='P&L RI Total')
##print(df_test.columns.tolist())

# 6) CAMPOS DE LA BASE FINAL CONSOLIDADA

In [34]:
# ===================================================================
# FUNCIONES DE FILTRADO CON FECHA Y GUARDADO SIMPLIFICADO
# ===================================================================

def filtrar_campos_consolidado_con_fecha(df_consolidado):
    """Filtra a exactamente 39 campos específicos incluyendo Fecha"""
    
    # CAMPOS OBJETIVO FINALES (39 campos)
    campos_objetivo = [
        "fecha",
        "escenario",
        "marca", 
        "lob_canal", 
        "no_mes_proyectado",
        "mes_proyectado",
        "pais", 
        "producto",
        "viaje",
        "partner",
        "orders",
        "gross_bookings",
        "up_front_incentives",
        "fees", 
        "commercial_discounts", 
        "cancellations",
        "cost_of_installments", 
        "credit_card_processing",
        "white_labels_api",
        "affiliates", 
        "income_from_outsourced_services", 
        "back_end_incentives", 
        "other_incentives", 
        "breakage_revenue",
        "media_revenue", 
        "revenue_tax",
        "customer_service",
        "errors",
        "frauds", 
        "customer_claims",
        "other_transactional_taxes", 
        "vendor_commissions",
        "intercompany_usd",
        "operations",
        "dif_fx",
        "hedge",
        "financial_results",
        "net_revenue",
        "fvm"
    ]
    
    print(f"\n🎯 **FILTRANDO A 39 CAMPOS ESPECÍFICOS**")
    print(f"  📊 Entrada: {df_consolidado.shape[0]:,} registros, {df_consolidado.shape[1]} columnas")
    
    # MAPEO MÍNIMO (solo si es necesario)
    #mapeo_nombres = {
        #'Fecha Mes': 'Mes Proyectado',  # Por si algún archivo usa este nombre
        #'Mes': 'Mes Proyectado',         # Por si algún archivo usa este nombre
        #'mes_proyectado': 'no_mes_proyectado',  # Por si algún archivo usa este nombr 
        #'currency_hedge': 'hedge',              # Por si algún archivo usa este nombrR  
        #'npv': 'fvm' 
    #}
    
    # Aplicar mapeo solo si es necesario
    #if any(col in df_consolidado.columns for col in mapeo_nombres.keys()):
        #df_consolidado = df_consolidado.rename(columns=mapeo_nombres)
        #print(f"  🔄 Mapeo aplicado")
    
    # IDENTIFICAR CAMPOS DISPONIBLES
    campos_disponibles = [campo for campo in campos_objetivo if campo in df_consolidado.columns]
    campos_faltantes = [campo for campo in campos_objetivo if campo not in df_consolidado.columns]
    
    print(f"  ✅ Disponibles: {len(campos_disponibles)}/{len(campos_objetivo)}")
    if campos_faltantes:
        print(f"  ⚠️  Faltantes: {campos_faltantes}")
    
    # CREAR CAMPOS FALTANTES
    for campo in campos_faltantes:
        if campo == "fecha":
            # Usar la columna Fecha del merge con factores
            if 'fecha_x' in df_consolidado.columns:
                df_consolidado["fecha"] = df_consolidado['fecha_x']
            elif 'fecha_y' in df_consolidado.columns:
                df_consolidado["fecha"] = df_consolidado['fecha_y']
            else:
                df_consolidado["fecha"] = pd.Timestamp.now().date()
            print(f"    ➕ Campo Fecha configurado")
        elif campo in ["escenario", "marca", "lob_canal", "producto", "partner"]:
            df_consolidado[campo] = ""
            print(f"    ➕ Campo dimensión: {campo} = ''")
        else:
            df_consolidado[campo] = 0.0
            print(f"    ➕ Campo métrica: {campo} = 0.0")

    # ACTUALIZAR no_mes_proyectado desde columna fecha
    if "fecha" in df_consolidado.columns:
        df_consolidado["no_mes_proyectado"] = pd.to_datetime(df_consolidado["fecha"], errors="coerce").dt.month
    
    # FILTRAR SOLO LOS 39 CAMPOS
    df_filtrado = df_consolidado[campos_objetivo].copy()
    
    print(f"  📊 Salida: {df_filtrado.shape[0]:,} registros, {df_filtrado.shape[1]} columnas")
    print(f"  ✅ Filtrado completado")
    
    return df_filtrado




def verificar_disclaimer_aplicado(df_consolidado):
    """
    Verifica que el disclaimer se haya aplicado correctamente
    para reconocimiento de ingresos
    """
    print(f"\n⚠️  **VERIFICANDO DISCLAIMER APLICADO (RECONOCIMIENTO INGRESOS):**")
    print("-" * 50)
    
    # Buscar registros de Paraguay y Uruguay
    registros_disclaimer = df_consolidado[
        df_consolidado['pais'].isin(['Paraguay', 'Uruguay'])
    ]
    
    if not registros_disclaimer.empty:
        print(f"✅ DISCLAIMER VERIFICADO:")
        for pais in ['Paraguay', 'Uruguay']:
            count = len(registros_disclaimer[registros_disclaimer['pais'] == pais])
            if count > 0:
                print(f"   {pais}: {count:,} registros usando factores de 'Other Countries'")
    else:
        print(f"📝 NOTA: No se encontraron registros de Paraguay o Uruguay")
        print(f"   Esto puede indicar que:")
        print(f"   - No había datos de estos países en las proyecciones originales")
        print(f"   - Los datos se filtraron durante el procesamiento")
    
    # Mostrar países únicos en el resultado final
    paises_unicos = sorted(df_consolidado['pais'].unique())
    print(f"\n📍 PAÍSES ÚNICOS EN RESULTADO FINAL:")
    print(f"   Total: {len(paises_unicos)} países")
    print(f"   Lista: {paises_unicos}")


def guardar_base_consolidada_unica(df_consolidado, timestamp, ruta_salida):
    """
    Guarda únicamente la base consolidada en un archivo CSV
    
    Args:
        df_consolidado: DataFrame con todos los datos consolidados
        timestamp: timestamp para el nombre del archivo
        ruta_salida: ruta donde guardar el archivo
    
    Returns:
        str: nombre del archivo guardado
    """
    try:
        # Crear directorio si no existe
        os.makedirs(ruta_salida, exist_ok=True)
        
        # Nombre del archivo
        nombre_archivo = f"base_consolidada_diaria_GD_{timestamp}.csv"
        ruta_completa = os.path.join(ruta_salida, nombre_archivo)
        
        # Guardar archivo CSV
        df_consolidado.to_csv(ruta_completa, index=False, encoding='utf-8-sig')
        
        print(f"  ✅ Archivo guardado: {nombre_archivo}")
        print(f"  📁 Ubicación: {ruta_salida}")
        print(f"  📊 Registros: {df_consolidado.shape[0]:,}")
        print(f"  📋 Columnas: {df_consolidado.shape[1]}")
        
        return nombre_archivo
        
    except Exception as e:
        print(f"  ❌ Error al guardar: {str(e)}")
        return None

# 7) PROCESAR TODOS LOS NEGOCIOS

In [35]:
# ===================================================================
# PROCESAR TODOS LOS NEGOCIOS
# ===================================================================

print("\n🚀 INICIANDO PROCESAMIENTO DE TODOS LOS NEGOCIOS...")
print("=" * 60)

resultados_negocios = {}

for negocio, config in configuracion_negocios.items():
    resultado = procesar_negocio(negocio, config)
    if resultado is not None:
        resultados_negocios[negocio] = resultado
        print(f"\n✅ {config['nombre_corto']} completado exitosamente")
    else:
        print(f"\n❌ {config['nombre_corto']} falló")

print(f"\n📊 RESUMEN DE CARGA:")
print(f"✅ Negocios procesados exitosamente: {len(resultados_negocios)}/{len(configuracion_negocios)}")
for negocio in resultados_negocios.keys():
    config = configuracion_negocios[negocio]
    resultado = resultados_negocios[negocio]
    print(f"  📁 {config['nombre_corto']}: {resultado['data'].shape[0]:,} registros, {len(resultado['metricas'])} métricas")


🚀 INICIANDO PROCESAMIENTO DE TODOS LOS NEGOCIOS...

🔄 PROCESANDO: White Labels / B2B2C
--------------------------------------------------
  📈 Cargando proyecciones desde 'P&L'...
    ✅ Dimensiones: (13230, 41)
    🔄 Mapeo lob_canal aplicado: API → B2B-MAY

  🔍 DIAGNÓSTICO DE COLUMNAS CRÍTICAS:
    📊 Total registros: 13,230
    📋 Columnas disponibles: ['Concatenado', 'escenario', 'mes_pivot', 'pais', 'lob_canal', 'marca', 'partner', 'viaje', 'producto', 'numero_mes_proyectado', 'mes_ri', 'anio', 'orders', 'gross_bookings', 'up_front_incentives', 'fees', 'commercial_discounts', 'income_from_outsourced_services', 'cancellations', 'cost_of_installments', 'credit_card_processing', 'white_labels_api', 'other_incentives', 'revenue_tax', 'back_end_incentives', 'breakage_revenue', 'media_other_revenue', 'errors', 'other_transactional_taxes', 'customer_claims', 'customer_service', 'vendor_commissions', 'intercompany_usd', 'operations', 'channels', 'frauds', 'efecto_financiero', 'dif_fx', 'curre

In [36]:
# DIAGNÓSTICO: buscar filas de agosto sin fecha en B2B2C
if 'B2B2C' in resultados_negocios:
    df_wl = resultados_negocios['B2B2C']['data']
    df_agosto = df_wl[df_wl['Mes_Numerico'] == 8].copy()
    
    sin_fecha = df_agosto['fecha'].isna()
    print(f"Filas agosto WL sin fecha: {sin_fecha.sum():,}")
    print(f"GB agosto TOTAL en output: {df_agosto['gross_bookings'].sum():,.0f}")
    print(f"GB agosto ORIGINAL (archivo): 81,874,469")
    print()

    # Ver cuánto GB se pierde por grupos con factor = 0
    suma_factores = df_agosto.groupby(['pais', 'partner', 'viaje'])['Factor combinado'].sum()
    grupos_cero = suma_factores[suma_factores == 0].index

    df_agosto['grupo_factor_cero'] = df_agosto[['pais','partner','viaje']].apply(tuple, axis=1).isin(grupos_cero)

    gb_perdido = df_agosto[df_agosto['grupo_factor_cero']]['gross_bookings'].sum()
    print(f"GB agosto perdido por factor=0: {gb_perdido:,.0f}")
    print(f"Diferencia observada:           {81_874_469 - 81_519_988:,.0f}")
    print()
    print("Combinaciones con factor=0:")
    print(suma_factores[suma_factores == 0])

Filas agosto WL sin fecha: 0
GB agosto TOTAL en output: 0
GB agosto ORIGINAL (archivo): 81,874,469

GB agosto perdido por factor=0: 0
Diferencia observada:           354,481

Combinaciones con factor=0:
Series([], Name: Factor combinado, dtype: float64)


# 8) GENERAR DISTRIBUCIÓN

In [37]:
# ===================================================================
# VALIDAR DISTRIBUCIÓN Y GENERAR ESTADÍSTICAS
# ===================================================================

print("\n✅ VALIDANDO DISTRIBUCIÓN DIARIA...")
print("=" * 50)

resultados_finales = {}

for negocio, resultado in resultados_negocios.items():
    config = resultado['config']
    df_data = resultado['data'].copy()
    metricas = resultado['metricas']
    
    print(f"\n🔧 VALIDANDO: {config['descripcion']}")
    print("-" * 40)
    
    if len(metricas) == 0:
        print(f"  ⚠️  No hay métricas para {config['nombre_corto']}")
        continue
    
    # MUESTRA DE DATOS (distribución ya aplicada en procesar_negocio)
    print(f"  📋 MUESTRA DE DATOS (distribución ya aplicada):")
    cols_muestra = ['pais', 'viaje'] + metricas[:2] + ['Factor combinado']
    cols_disponibles = [col for col in cols_muestra if col in df_data.columns]
    if cols_disponibles:
        print(df_data[cols_disponibles].head(2).to_string(index=False))

    # 🔍 DIAGNÓSTICO DE FACTORES Y DUPLICADOS
    print(f"\n  🔍 DIAGNÓSTICO DE FACTORES Y DUPLICADOS:")
    print(f"    📊 Total registros en df_data: {len(df_data):,}")
    
    # Verificar duplicados
    cols_clave = ['pais', 'viaje', 'fecha', 'anio']
    if 'partner' in df_data.columns:
        cols_clave.append('partner')
    if 'mes_proyectado' in df_data.columns:
        cols_clave.append('mes_proyectado')
    elif 'mes' in df_data.columns:
        cols_clave.append('mes')
    
    duplicados = df_data.duplicated(subset=cols_clave).sum()
    print(f"    🔄 Registros duplicados: {duplicados:,}")
    
    # Verificar factores y cobertura
    registros_con_factor = ~df_data['Factor combinado'].isna()
    total_registros = len(df_data)
    registros_factor = registros_con_factor.sum()

    if 'Factor combinado' in df_data.columns:
        print(f"    ✅ Registros con factor: {registros_factor:,}/{total_registros:,}")
        print(f"    📈 Rango factores: {df_data['Factor combinado'].min():.6f} - {df_data['Factor combinado'].max():.6f}")
        print(f"    📊 Suma total factores: {df_data['Factor combinado'].sum():.2f}")
        
        # Verificar suma de factores por grupo mes/país/viaje
        if len(df_data) > 0:
            cols_agrup = []
            if 'mes_proyectado' in df_data.columns:
                cols_agrup.append('mes_proyectado')
            elif 'mes' in df_data.columns:
                cols_agrup.append('mes')
            cols_agrup.extend(['anio', 'pais', 'viaje'])
            
            if all(col in df_data.columns for col in cols_agrup):
                suma_factores = df_data.groupby(cols_agrup)['Factor combinado'].sum()
                print(f"    🎯 Suma factores por grupo ({'+'.join(cols_agrup)}):")
                print(f"       Min: {suma_factores.min():.4f}, Max: {suma_factores.max():.4f}")
                print(f"       Media: {suma_factores.mean():.4f}")
                grupos_incorrectos = ((suma_factores < 0.95) | (suma_factores > 1.05)).sum()
                print(f"    ⚠️  Grupos con suma ≠ 1.0: {grupos_incorrectos}/{len(suma_factores)}")
                
                if grupos_incorrectos > 0:
                    problematicos = suma_factores[(suma_factores < 0.95) | (suma_factores > 1.05)]
                    print(f"    🔍 Ejemplos problemáticos:")
                    for idx, valor in problematicos.head(3).items():
                        print(f"       {idx}: {valor:.4f}")

    # Verificar métricas distribuidas
    metricas_diagnostico = [m for m in ['gross_bookings', 'orders', 'fees'] if m in df_data.columns]
    if not metricas_diagnostico:
        metricas_diagnostico = metricas[:3]
    print(f"    💰 MÉTRICAS DISTRIBUIDAS (suma total):")
    for metrica in metricas_diagnostico:
        if metrica in df_data.columns:
            print(f"       {metrica}: {df_data[metrica].sum():,.2f}")

    # AGREGAR METADATOS FINALES
    df_data['Factor_Aplicado'] = df_data['Factor combinado']
    df_data['Registros_Con_Factor'] = registros_con_factor
    df_data['Cobertura_Factor'] = (registros_factor / total_registros * 100) if total_registros > 0 else 0
    
    resultados_finales[negocio] = {
        'data': df_data,
        'metricas': metricas,
        'config': config,
        'stats': {
            'total_registros': total_registros,
            'registros_con_factor': registros_factor,
            'cobertura': registros_factor / total_registros * 100 if total_registros > 0 else 0
        }
    }
    
    print(f"  ✅ {config['nombre_corto']} validado exitosamente")

print(f"\n📊 RESUMEN DE DISTRIBUCIÓN:")
for negocio, resultado in resultados_finales.items():
    config = resultado['config']
    stats = resultado['stats']
    print(f"  {config['nombre_corto']}: {stats['registros_con_factor']:,}/{stats['total_registros']:,} ({stats['cobertura']:.1f}%)")



✅ VALIDANDO DISTRIBUCIÓN DIARIA...

🔧 VALIDANDO: White Labels / B2B2C
----------------------------------------
  📋 MUESTRA DE DATOS (distribución ya aplicada):
     pais    viaje    orders  gross_bookings  Factor combinado
Argentina Domestic 10.110219    10291.822554          0.039987
Argentina Domestic 10.287155    10471.936750          0.040687

  🔍 DIAGNÓSTICO DE FACTORES Y DUPLICADOS:
    📊 Total registros en df_data: 400,680
    🔄 Registros duplicados: 351,920
    ✅ Registros con factor: 400,680/400,680
    📈 Rango factores: 0.000000 - 0.202765
    📊 Suma total factores: 13230.00
    🎯 Suma factores por grupo (mes+anio+pais+viaje):
       Min: 7.0000, Max: 49.0000
       Media: 24.5000
    ⚠️  Grupos con suma ≠ 1.0: 84/84
    🔍 Ejemplos problemáticos:
       (1.0, 2027, 'Argentina', 'Domestic'): 14.0000
       (1.0, 2027, 'Argentina', 'International'): 14.0000
       (1.0, 2027, 'Brasil', 'Domestic'): 42.0000
    💰 MÉTRICAS DISTRIBUIDAS (suma total):
       gross_bookings: 746,15

# 9) CONSOLIDAR RESULTADOS

In [38]:
# ===================================================================
# CONSOLIDAR TODOS LOS RESULTADOS
# ===================================================================

if len(resultados_finales) > 0:
    print(f"\n📋 CONSOLIDANDO {len(resultados_finales)} NEGOCIOS...")
    print("-" * 40)
    
    # Combinar todos los DataFrames
    dfs_consolidar = []
    for negocio, resultado in resultados_finales.items():
        df_negocio = resultado['data'].copy()
        df_negocio['Fuente_Negocio'] = negocio
        dfs_consolidar.append(df_negocio)
    
    df_consolidado = pd.concat(dfs_consolidar, ignore_index=True)

    # mes_proyectado: mes_venta para API/HTML, fallback a mes_ri para WLs
    mapeo_meses_nombre = {
        1: 'Enero',      2: 'Febrero',   3: 'Marzo',      4: 'Abril',
        5: 'Mayo',       6: 'Junio',     7: 'Julio',      8: 'Agosto',
        9: 'Septiembre', 10: 'Octubre',  11: 'Noviembre', 12: 'Diciembre'
    }
    if 'mes_venta' in df_consolidado.columns:
        df_consolidado['mes_source'] = df_consolidado['mes_venta'].fillna(df_consolidado['mes_ri'])
    else:
        df_consolidado['mes_source'] = df_consolidado['mes_ri']

    df_consolidado['mes_proyectado'] = df_consolidado['mes_source'].map(mapeo_meses_nombre)
    df_consolidado.drop(columns=['mes_source'], inplace=True, errors='ignore')
    print(f"  ✅ mes_proyectado normalizado (mes_venta/mes_ri): {sorted(df_consolidado['mes_proyectado'].dropna().unique())}")

    print(f"  ✅ Consolidado: {df_consolidado.shape[0]:,} registros")
    print(f"  📊 Negocios: {', '.join(df_consolidado['Tipo_Negocio'].unique())}")
    print(f"  🌍 Países: {df_consolidado['pais'].nunique()}")
    print(f"  🎯 Tipos de viaje: {df_consolidado['viaje'].nunique()}")

    # ===================================================================
    # CALCULAR NET_REVENUE Y FVM  ← PRIMERO (antes del filtro)
    # ===================================================================
    print(f"\n📐 CALCULANDO NET_REVENUE Y FVM...")
    print("-" * 40)

    # NET REVENUE
    cols_net_revenue = [
        'up_front_incentives',
        'fees',
        'commercial_discounts',
        'cancellations',
        'income_from_outsourced_services',
        'back_end_incentives',
        'other_incentives',
        'breakage_revenue',
        'media_revenue',
        'revenue_tax'
    ]
    cols_nr_disponibles = [c for c in cols_net_revenue if c in df_consolidado.columns]
    df_consolidado['net_revenue'] = df_consolidado[cols_nr_disponibles].sum(axis=1)
    print(f"  ✅ net_revenue calculado con {len(cols_nr_disponibles)}/{len(cols_net_revenue)} componentes")
    if len(cols_nr_disponibles) < len(cols_net_revenue):
        print(f"  ⚠️  Componentes faltantes: {set(cols_net_revenue) - set(cols_nr_disponibles)}")

    # FVM (Financial Variable Margin)
    cols_fvm = [
        'net_revenue',
        'cost_of_installments',
        'credit_card_processing',
        'white_labels_api',
        'affiliates',
        'customer_service',
        'errors',
        'frauds',
        'intercompany_usd',
        'customer_claims',
        'other_transactional_taxes',
        'vendor_commissions',
        'dif_fx',
        'hedge',
        'financial_results'
    ]
    cols_fvm_disponibles = [c for c in cols_fvm if c in df_consolidado.columns]
    df_consolidado['fvm'] = df_consolidado[cols_fvm_disponibles].sum(axis=1)
    print(f"  ✅ fvm calculado con {len(cols_fvm_disponibles)}/{len(cols_fvm)} componentes")
    if len(cols_fvm_disponibles) < len(cols_fvm):
        print(f"  ⚠️  Componentes faltantes: {set(cols_fvm) - set(cols_fvm_disponibles)}")

    print(f"  📊 net_revenue total: {df_consolidado['net_revenue'].sum():,.2f}")
    print(f"  📊 fvm total: {df_consolidado['fvm'].sum():,.2f}")

    # ===================================================================
    # ELIMINAR LÍNEAS DONDE TODAS LAS MÉTRICAS CLAVE SON CERO  ← DESPUÉS
    # ===================================================================
    print(f"\n🚫 ELIMINANDO REGISTROS SIN ACTIVIDAD FINANCIERA...")
    print("-" * 40)

    mask_todo_cero = (
        (df_consolidado['gross_bookings'] == 0) &
        (df_consolidado['up_front_incentives'] == 0) &
        (df_consolidado['fees'] == 0) &
        (df_consolidado['net_revenue'] == 0) &
        (df_consolidado['fvm'] == 0)
    )
    registros_antes = len(df_consolidado)
    df_consolidado = df_consolidado[~mask_todo_cero]
    registros_despues = len(df_consolidado)

    print(f"  📊 Registros antes: {registros_antes:,}")
    print(f"  📊 Registros después: {registros_despues:,}")
    print(f"  🚫 Registros eliminados (todo cero): {registros_antes - registros_despues:,}")
    print(f"  ✅ Filtro aplicado exitosamente")


    # ⚠️ APLICAR DISCLAIMER AL FINAL (después del merge con factores)
    df_consolidado = aplicar_disclaimer_pais(df_consolidado, 'pais')
    print("✅ Disclaimer aplicado post-merge")
    # VERIFICAR DISCLAIMER APLICADO
    verificar_disclaimer_aplicado(df_consolidado)

    # FILTRAR A 39 CAMPOS ESPECÍFICOS CON FECHA
    df_consolidado = filtrar_campos_consolidado_con_fecha(df_consolidado)

else:
    print(f"\n❌ No hay resultados para consolidar")
    df_consolidado = None


📋 CONSOLIDANDO 3 NEGOCIOS...
----------------------------------------
  ✅ mes_proyectado normalizado (mes_venta/mes_ri): ['Diciembre', 'Enero', 'Febrero', 'Marzo', 'Noviembre', 'Octubre', 'Septiembre']
  ✅ Consolidado: 478,272 registros
  📊 Negocios: WL_B2B2C, API, HTML
  🌍 Países: 8
  🎯 Tipos de viaje: 2

📐 CALCULANDO NET_REVENUE Y FVM...
----------------------------------------
  ✅ net_revenue calculado con 9/10 componentes
  ⚠️  Componentes faltantes: {'media_revenue'}
  ✅ fvm calculado con 13/15 componentes
  ⚠️  Componentes faltantes: {'financial_results', 'hedge'}
  📊 net_revenue total: 135,575,891.81
  📊 fvm total: 38,567,051.53

🚫 ELIMINANDO REGISTROS SIN ACTIVIDAD FINANCIERA...
----------------------------------------
  📊 Registros antes: 478,272
  📊 Registros después: 144,228
  🚫 Registros eliminados (todo cero): 334,044
  ✅ Filtro aplicado exitosamente
✅ Disclaimer aplicado post-merge

⚠️  **VERIFICANDO DISCLAIMER APLICADO (RECONOCIMIENTO INGRESOS):**
----------------------

# 10) GUARDADO DE OUTPUT DIARIO

In [39]:
# ===================================================================
# GUARDAR ÚNICAMENTE BASE CONSOLIDADA
#guardar en carpeta nueva de Anto
# ===================================================================

if df_consolidado is not None:
    print("\n💾 GUARDANDO BASE CONSOLIDADA...")
    
    # Crear timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    
    ruta_salida = r"C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Run Rate/2026.09.14 - W37/Distribucion Diaria"
    
    # Guardar únicamente la base consolidada
    archivo_guardado = guardar_base_consolidada_unica(df_consolidado, timestamp, ruta_salida)
    
    if archivo_guardado:
        print(f"\n🎯 PROCESO COMPLETADO")
        print("=" * 50)
        print(f"✅ Archivo: {archivo_guardado}")
        print(f"✅ Formato: CSV")
        print(f"✅ Contiene: B2B2C + B2B-MAY + B2B-MIN")
        print(f"✅ Distribución diaria aplicada")
        print(f"✅ Filtro gross_bookings != 0 aplicado")
        print(f"✅ 39 campos específicos")
        print(f"✅ Fechas desde {df_consolidado['fecha'].min()} hasta {df_consolidado['fecha'].max()}")
    else:
        print("❌ Error al guardar el archivo")
        
else:
    print("❌ No hay datos para guardar")



💾 GUARDANDO BASE CONSOLIDADA...
  ✅ Archivo guardado: base_consolidada_diaria_GD_20260924_1253.csv
  📁 Ubicación: C:/Users/antonella.difranco/despegar365/Control de Gestión - Documentos/Planeamiento/2026-27/B2B & WLs/Run Rate/2026.09.14 - W37/Distribucion Diaria
  📊 Registros: 144,228
  📋 Columnas: 39

🎯 PROCESO COMPLETADO
✅ Archivo: base_consolidada_diaria_GD_20260924_1253.csv
✅ Formato: CSV
✅ Contiene: B2B2C + B2B-MAY + B2B-MIN
✅ Distribución diaria aplicada
✅ Filtro gross_bookings != 0 aplicado
✅ 39 campos específicos
✅ Fechas desde 2026-09-01 00:00:00 hasta 2027-03-31 00:00:00
